# 3 · Dynamic tools

"Dynamic tools" means the tool surface is **not fixed at compile time**. The
`ToolRegistry` is just an in-memory map: you can register, replace, and unregister
tools **while the agent is alive**. New capabilities can be bolted on without
recompiling — exactly how a running agent gets a new skill, or how a user exposes
a private function as a callable tool.

This demo is **fully local**. We start with only the built-ins, then:
1. register a brand-new custom tool at runtime,
2. replace an existing tool's handler,
3. observe the registry's tool list change before/after.


In [ ]:
:dep agent_loop = { path = "/home/christian/Sandbox/agent-loop" }
:dep serde_json = "1"

use agent_loop::tools::{ToolRegistry, Tool, ToolResult, s_required, builtin_tools};
use serde_json::{json, Value};

let mut registry = ToolRegistry::with_builtins();
println!("initial tools ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));


### Register a new tool at runtime

Here we add a `weather` tool that didn't exist a moment ago. It takes a city and
returns a canned forecast. The agent running in the same process could call it
immediately on the next turn.


In [ ]:

// A tool that did NOT exist at compile time; added live.
registry.register(Tool::new(
    "weather",
    "Return a weather forecast for a given city.",
    s_required(json!({"city": {"type":"string"}}), &["city"]),
    |args| {
        let city = args.get("city").and_then(Value::as_str).unwrap_or("unknown");
        Ok(ToolResult::ok(format!("sunny, 22°C in {city}")))
    },
));

println!("after adding weather ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));
println!("can call it now -> {}", registry.get("weather").unwrap()
    .run(&json!({"city": "Berlin"})).unwrap().output);


### Replace a handler at runtime

Same name, new behaviour: drop-in upgrades without changing the call site. The
model never knows — it just continues emitting `weather(...)` calls.


In [ ]:

let before = registry.get("weather").unwrap().run(&json!({"city":"Oslo"})).unwrap().output;

// Same tool name, new handler.
registry.register(Tool::new(
    "weather",
    "Return a weather forecast for a given city.",
    s_required(json!({"city": {"type":"string"}}), &["city"]),
    |args| {
        let city = args.get("city").and_then(Value::as_str).unwrap_or("unknown");
        Ok(ToolResult::ok(format!("10 cm of snow in {city} (winter storm)")))
    },
));

let after = registry.get("weather").unwrap().run(&json!({"city":"Oslo"})).unwrap().output;
println!("before replacement: {before}");
println!("after  replacement: {after}");


### Unregister a tool

Removing a capability is just as easy. Once unregistered, calls to it report the
tool as unknown so the model can adapt.


In [ ]:

registry.unregister("weather");
println!("after removing weather ({}): {}", registry.len(),
    registry.all().iter().map(|t| t.name.as_str()).collect::<Vec<_>>().join(", "));

let unknown = agent_loop::chat::ToolCall { id: "x".into(), name: "weather".into(), arguments: "{}".into() };
let msg = agent_loop::executor::execute_one(&registry, &unknown);
println!("call to removed tool -> {}", msg.content.unwrap());


### Why this matters

The request in demo 1 serializes whatever is in the registry at request time
(`registry.as_chat_tools()`). So adding a tool dynamically *changes the next
request* the model sees — the model learns about the new tool on the very next
turn. Dynamic + parallel + the loop from demo 1 = a live, growing agent.
